# Project 2 Part I (Core)

*Project 2: Initial Analysis and Problem Selection*

---

## Objective

Perform an initial Exploratory Data Analysis (EDA) on at least four datasets, identify and diagnose potential issues, and select a specific problem to address (regression, classification, clustering, or forecasting). Submit a repository containing the selected dataset, the initial EDA, and a description of the chosen problem.

---

# Part I: Dataset Search and Analysis

## Problem Statement

# Dataset Description

## Dataset Source

In this project, we analyze the "" dataset, obtained from "". The dataset contains information about "", including features such as "".

The dataset can be accessed and downloaded from the following link:
link_here

## Dataset Information

In [ ]:
print(f"Number of observations: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")

- Target variable: `target`

## Data Dictionary

Insert dictionary here

---

# Data Loading and Inspection

## Import Libraries

In [ ]:
# Standard library imports
import math
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

### Dataset Availability and Path Setup

In [ ]:
data_dir = Path("../datasets")

# List available files (for verification)
available_files = list(data_dir.iterdir())
print("Available files:", available_files)
file_name = "dataset_name.csv"
dataset_path = data_dir / file_name

In [ ]:
# Validate dataset existence
if not dataset_path.exists():
    raise FileNotFoundError(f"{file_name} not found in {data_dir}")

print("Dataset found at:", dataset_path)

## Load Dataset

In [ ]:
df = pd.read_csv(dataset_path)

## Dataset Overview


### First and Last Records

In [ ]:
df.head()

In [ ]:
df.tail()

### Verify Structure

In [ ]:
print(f"{file_name}")
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} rows, {df.shape[1]} columns\n")

---

## Initial Data Inspection

### Inspect Data Types

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

### Descriptive Statistics

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

---

### Missing Values

In [ ]:
df.isnull().sum()

Visualization:

In [ ]:
fig = px.imshow(
    df.isnull(),
    aspect="auto",
    labels=dict(x="Variables", y="Observations", color="Missing"),
    title="Missing Values Heatmap"
)

fig.update_layout(height=500)
fig.show()

### Duplicate Records Check

In [ ]:
n_duplicated = df.duplicated().sum()
print(f"Exact duplicated rows before cleaning: {n_duplicated}")

---

## Distribution Analysis

Numeric:

Histograms

In [ ]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

# Dashboard layout
n_cols = 4
n_rows = math.ceil(len(numeric_columns) / n_cols)

# Create subplot figure
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=numeric_columns
)

# Add histograms
for i, col in enumerate(numeric_columns):
    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Histogram(
            x=df[col],
            name=col,
            nbinsx=30
        ),
        row=row,
        col=col_pos
    )

# Update layout
fig.update_layout(
    title="Interactive Histogram Dashboard — Numeric Features",
    height=300 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white",
    bargap=0.05
)

# Descriptive axis labels
fig.update_xaxes(title_text="Feature Values")
fig.update_yaxes(title_text="Number of Observations")

# Show dashboard
fig.show()

**Analysis**

Categorical:

### Categorical Value Consistency

In [ ]:
categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"{'='*40}")

    summary = (
        df[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    summary.columns = [col, 'Count']

    summary['Percentage'] = (
        summary['Count'] / len(df) * 100
    ).round(2)

    print(summary)

In [ ]:
MAX_CATEGORIES = 10

n_cols = 2
n_rows = math.ceil(len(categorical_cols) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=categorical_cols,
    vertical_spacing=0.10
)

for i, col in enumerate(categorical_cols):

    summary = (
        df[col]
        .value_counts(dropna=False)
        .head(MAX_CATEGORIES)
        .reset_index()
    )

    summary.columns = [col, "Count"]

    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Bar(
            x=summary[col].astype(str),
            y=summary["Count"],
            text=summary["Count"],
            textposition="outside",
            name=col
        ),
        row=row,
        col=col_pos
    )

fig.update_layout(
    title="Categorical Variables Distribution Dashboard",
    height=400 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white"
)

fig.show()

**Analysis**

countplots

### Outlier Detection

In [ ]:
if len(numeric_columns) > 0:
    # create boxplots

    # Select numeric columns
    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    # Dashboard layout
    n_cols = 4
    n_rows = math.ceil(len(numeric_columns) / n_cols)

    # Create subplot figure
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=numeric_columns
    )

    # Add boxplots
    for i, col in enumerate(numeric_columns):
        row = (i // n_cols) + 1
        col_pos = (i % n_cols) + 1

        fig.add_trace(
            go.Box(
                y=df[col],
                name=col,
                boxmean=True
            ),
            row=row,
            col=col_pos
        )

    # Update layout
    fig.update_layout(
        title="Interactive Boxplot Dashboard — Numeric Features",
        height=300 * n_rows,
        width=1400,
        showlegend=False,
        template="plotly_white"
    )

    # Add axis labels
    fig.update_yaxes(title_text="Observed Values")

    # Show dashboard
    fig.show()

else:
    print("No numeric variables available for outlier analysis.")

**Analysis**

IQR calculation.

In [ ]:
feature_columns = df.select_dtypes(
    include=np.number
).columns

outlier_summary = []

for col in feature_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_summary.append({
        "Feature": col,
        "Outliers": n_outliers,
        "Percentage": round(n_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

---

# Exploratory Data Analysis (EDA)

### Correlation Analysis

In [ ]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

corr_matrix = df[numeric_columns].corr()

corr_matrix

### Heatmap visualization

In [ ]:
# Correlation matrix
corr_matrix = df[numeric_columns].corr()

# Create mask for upper triangle
mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool)
)

# Apply mask
corr_matrix_masked = corr_matrix.mask(mask)

fig = px.imshow(
    corr_matrix_masked,
    text_auto=".2f",
    color_continuous_scale="BuGn",
    aspect="auto",
    title="Correlation Matrix Heatmap"
)

fig.update_layout(
    width=900,
    height=800
)

fig.show()

---

## Potential Target Variable

Candidate target:
- target_column

Potential ML task:
- Regression / Classification / Clustering / Forecasting

Rationale:
Brief explanation.

---

# EDA Findings Summary

## Findings

- Dataset contains 5% missing values.
- Strong correlation between X and Y.
- Several outliers in feature Z.
- Suitable for regression because target variable is continuous.

---

### Acknowledgements